# Task 1.2 - Data Cleaning & EDA
**Skill Set Go EduTech - AI/ML Internship, Week 1**

**Dataset:** UCI Wine Recognition data (178 chemical analyses of wines grown in the
same region of Italy, derived from three different cultivars), loaded offline from
`sklearn.datasets.load_wine()` and re-exported as a realistic messy CSV by
`make_raw_dataset.py`. The injected formatting problems are documented in that
script and in the README, so every cleaning decision below is defensible.

**Question this notebook answers:** *which chemical measurements separate the three
cultivars, and what has to be fixed in the raw export before any model can use it?*

**Plan:** inspect -> assess quality -> clean with written reasoning -> explore ->
interpret -> save the cleaned dataset separately from the raw file.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend; remove this line when running in Jupyter
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
VIZ = "visualizations"
os.makedirs(VIZ, exist_ok=True)

raw = pd.read_csv("dataset/wine_quality_raw.csv")
print("shape:", raw.shape)
raw.head()

## 1. First inspection
Before changing anything: what columns exist, what types did pandas infer, and how
much of each column is actually populated?

In [ ]:
print(raw.info())
print("\n--- numeric summary ---")
print(raw.describe().T[["count", "mean", "min", "max"]])

In [ ]:
missing = raw.isna().sum()
missing_pct = (missing / len(raw) * 100).round(2)
quality = pd.DataFrame({"missing": missing, "missing_%": missing_pct,
                        "dtype": raw.dtypes.astype(str),
                        "unique": raw.nunique()})
print(quality[quality["missing"] > 0])
print("\nexact duplicate rows:", raw.duplicated().sum())
print("duplicate sample_id values:", raw["sample_id"].duplicated().sum())

### What the inspection found
1. `proline` came in as **object**, not a number - it holds strings like `"1,065 mg/L"`.
2. `alcohol`, `magnesium` and `hue` have missing cells.
3. There are **exact duplicate rows**, confirmed by duplicated `sample_id` values.
4. `cultivar` has far more distinct values than the three that should exist.
5. `magnesium` has a negative minimum and `ash` has a zero minimum - both are
   chemically impossible, so they are bad data rather than genuine extremes.

In [ ]:
print("distinct cultivar labels in the raw file:", raw["cultivar"].nunique())
print(raw["cultivar"].value_counts())

## 2. Cleaning, with the reason recorded for every decision
The raw DataFrame is never modified - all work happens on a copy, so `dataset/wine_quality_raw.csv`
stays untouched.

In [ ]:
df = raw.copy()
log = []  # every action is appended here and printed at the end

### 2.1 Duplicates
**Decision: drop exact duplicates.** These are a re-export artefact - the whole row,
including `sample_id`, repeats. Keeping them would double-count real samples and
inflate any correlation computed later.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
log.append(f"dropped {before - len(df)} exact duplicate rows ({before} -> {len(df)})")
print(log[-1])

### 2.2 `proline` stored as text
**Decision: strip the unit and thousands separator, then convert to float.** The unit
is identical for every row (mg/L), so it carries no information and belongs in the
column name, not in each cell.

In [ ]:
df["proline"] = (df["proline"].astype(str)
                 .str.replace(",", "", regex=False)
                 .str.replace("mg/L", "", regex=False)
                 .str.strip())
df["proline"] = pd.to_numeric(df["proline"], errors="coerce")
df = df.rename(columns={"proline": "proline_mg_l"})
log.append("converted proline text -> numeric, renamed to proline_mg_l")
print(log[-1], "| dtype now:", df["proline_mg_l"].dtype,
      "| failed conversions:", df["proline_mg_l"].isna().sum())

### 2.3 Inconsistent category text
**Decision: normalise whitespace and case, then map the abbreviation `Cult. X` onto
the full label.** These are the same three cultivars recorded inconsistently; leaving
them apart would produce 12 fake classes.

In [ ]:
cleaned_label = (df["cultivar"].str.strip().str.lower()
                 .str.replace(r"\s+", " ", regex=True)
                 .str.replace("cult.", "cultivar", regex=False)
                 .str.replace(r"\s+", " ", regex=True)
                 .str.title())
df["cultivar"] = cleaned_label
log.append(f"normalised cultivar labels: {raw['cultivar'].nunique()} -> {df['cultivar'].nunique()} categories")
print(log[-1])
print(df["cultivar"].value_counts())

### 2.4 Mixed date formats
**Decision: parse both `YYYY-MM-DD` and `DD/MM/YYYY` into one datetime column.**
`format="mixed"` with `dayfirst=True` resolves the ambiguous `12/02/2026` style
correctly for this export.

In [ ]:
df["batch_date"] = pd.to_datetime(df["batch_date"], format="mixed", dayfirst=True)
log.append("parsed batch_date into datetime64")
print(log[-1], "| range:", df["batch_date"].min().date(), "->", df["batch_date"].max().date())

### 2.5 Impossible values
**Decision: convert them to NaN rather than delete the rows.** A negative magnesium
reading or a zero ash reading is a recording error in *one cell*; the other twelve
measurements in that row are still valid, so throwing the row away loses good data.
Marking the cell as missing lets the next step handle it consistently.

In [ ]:
bad_mg = (df["magnesium"] <= 0).sum()
bad_ash = (df["ash"] <= 0).sum()
df.loc[df["magnesium"] <= 0, "magnesium"] = np.nan
df.loc[df["ash"] <= 0, "ash"] = np.nan
log.append(f"nulled {bad_mg} impossible magnesium and {bad_ash} impossible ash values")
print(log[-1])

### 2.6 Missing values - handled per column, not with one blanket rule
The guide warns against filling everything the same way, so each column is decided
on its meaning:

| column | missing | decision | reason |
|---|---|---|---|
| `alcohol` | ~4% | **median within the same cultivar** | alcohol content differs systematically between cultivars, so a global median would drag values toward the overall centre and blur exactly the difference I want to study |
| `magnesium` | ~8% + nulled errors | **median within the same cultivar** | same argument; median is used instead of mean because the column is right-skewed with genuine high outliers |
| `hue` | ~6% | **median within the same cultivar** | same |
| `ash` | 2 nulled in 2.5 | **median within the same cultivar** | the two zero readings became NaN above and must be filled here, otherwise they silently break model fitting in Task 1.4 |

Group-wise median is chosen over dropping rows because dropping would remove roughly
a sixth of a 178-row dataset - too expensive for a dataset this small. The imputation
is flagged in a boolean column so downstream work can tell real from filled values.

In [ ]:
impute_cols = ["alcohol", "magnesium", "hue", "ash"]
df["was_imputed"] = df[impute_cols].isna().any(axis=1)

for col in impute_cols:
    n = df[col].isna().sum()
    df[col] = df[col].fillna(df.groupby("cultivar")[col].transform("median"))
    log.append(f"filled {n} missing '{col}' with the median of its cultivar")
    print(log[-1])

print("\nremaining missing values:", int(df.isna().sum().sum()))
print("rows containing at least one imputed cell:", int(df['was_imputed'].sum()))

### 2.7 Outlier check - inspect, do not automatically delete
The IQR rule is used to *find* extreme points, not to remove them. In a 178-sample
chemical dataset an extreme reading is usually a real wine, not an error.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
rows = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append({"column": col, "outliers": int(((df[col] < lo) | (df[col] > hi)).sum()),
                 "lower_fence": round(lo, 2), "upper_fence": round(hi, 2)})
outlier_table = pd.DataFrame(rows).sort_values("outliers", ascending=False)
print(outlier_table.head(8).to_string(index=False))
log.append("outliers flagged by IQR but KEPT - plausible chemistry, not recording errors")
print("\n" + log[-1])

## 3. Cleaning summary

In [ ]:
print(f"raw   : {raw.shape[0]} rows x {raw.shape[1]} cols")
print(f"clean : {df.shape[0]} rows x {df.shape[1]} cols\n")
for i, entry in enumerate(log, 1):
    print(f"{i:>2}. {entry}")

## 4. Exploratory Data Analysis
Each chart answers a stated question, and each is followed by a written interpretation.

### Q1. Are the three cultivars balanced?
Class balance decides whether accuracy is a safe metric later in Task 1.4.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["cultivar"].value_counts().sort_index()
ax.bar(counts.index, counts.values, color="#4c72b0")
for i, v in enumerate(counts.values):
    ax.text(i, v + 1, str(v), ha="center")
ax.set_title("Samples per cultivar")
ax.set_ylabel("count")
plt.tight_layout(); plt.savefig(f"{VIZ}/01_class_balance.png", dpi=120); plt.close()
print(counts)

**Interpretation.** The classes are mildly imbalanced (roughly 59 / 71 / 48). No class
is rare enough to need resampling, but a naive "always predict the largest class"
baseline would already score about 40%, so accuracy alone will not be convincing
evidence in the modelling task.

### Q2. How is each measurement distributed, and is it skewed?

In [ ]:
show = ["alcohol", "malic_acid", "magnesium", "color_intensity", "flavanoids", "proline_mg_l"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), show):
    ax.hist(df[col], bins=20, color="#55a868", edgecolor="white")
    ax.axvline(df[col].mean(), color="#c44e52", ls="--", lw=1.5, label="mean")
    ax.axvline(df[col].median(), color="#4c72b0", ls=":", lw=1.5, label="median")
    ax.set_title(col); ax.legend(fontsize=7)
fig.suptitle("Distributions of key measurements")
plt.tight_layout(); plt.savefig(f"{VIZ}/02_distributions.png", dpi=120); plt.close()
print(df[show].skew().round(2).sort_values(ascending=False))

**Interpretation.** `malic_acid`, `magnesium` and `color_intensity` are clearly
right-skewed - their means sit to the right of their medians. That is the concrete
justification for having imputed with the **median** in section 2.6: the mean of a
skewed column is pulled by its tail. `alcohol` and `flavanoids` are close to symmetric.

### Q3. Which measurements actually separate the cultivars?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, col in zip(axes, ["flavanoids", "color_intensity", "proline_mg_l"]):
    groups = [df.loc[df["cultivar"] == c, col] for c in sorted(df["cultivar"].unique())]
    ax.boxplot(groups, tick_labels=[c.replace("Cultivar ", "") for c in sorted(df["cultivar"].unique())])
    ax.set_title(f"{col} by cultivar"); ax.set_xlabel("cultivar")
fig.suptitle("Do the groups actually differ?")
plt.tight_layout(); plt.savefig(f"{VIZ}/03_boxplots_by_cultivar.png", dpi=120); plt.close()
print(df.groupby("cultivar")[["flavanoids", "color_intensity", "proline_mg_l", "alcohol"]]
      .median().round(2))

**Interpretation.** `flavanoids` separates the groups most cleanly - Cultivar C sits
far below the other two with barely overlapping boxes. `proline_mg_l` isolates
Cultivar A, whose median (1095) is far above Cultivar B (495) and C (628). `color_intensity` splits A and
C from B. No single measurement separates all three, but these three together very
nearly do, which predicts that even a simple classifier should perform well.

### Q4. Which measurements move together?
Strongly correlated inputs carry redundant information.

In [ ]:
corr = df[numeric_cols].drop(columns=["was_imputed"], errors="ignore").corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title("Correlation matrix")
plt.tight_layout(); plt.savefig(f"{VIZ}/04_correlation_heatmap.png", dpi=120); plt.close()

pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
         .stack().sort_values(key=abs, ascending=False).head(6))
print(pairs.round(2))

**Interpretation.** `total_phenols` and `flavanoids` correlate at about 0.86, and
`od280_od315_of_diluted_wines` tracks both. Chemically these measure overlapping
phenolic content, so they are near-duplicates as far as a model is concerned - worth
remembering when interpreting feature importances rather than treating each as an
independent signal. **Correlation here is co-occurrence, not causation:** high
flavanoids do not *cause* high total phenols, both reflect the same underlying grape
chemistry.

### Q5. Does the relationship between two measurements depend on the cultivar?

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
colors = {"Cultivar A": "#4c72b0", "Cultivar B": "#dd8452", "Cultivar C": "#55a868"}
for name, sub in df.groupby("cultivar"):
    ax.scatter(sub["flavanoids"], sub["color_intensity"], label=name,
               alpha=0.8, s=35, color=colors.get(name))
ax.set_xlabel("flavanoids"); ax.set_ylabel("color_intensity")
ax.set_title("Flavanoids vs colour intensity, coloured by cultivar"); ax.legend()
plt.tight_layout(); plt.savefig(f"{VIZ}/05_scatter_flavanoids_color.png", dpi=120); plt.close()
print(df.groupby("cultivar")[["flavanoids", "color_intensity"]].agg(["mean", "std"]).round(2))

**Interpretation.** The three cultivars occupy three largely distinct regions of this
two-dimensional space: Cultivar C is low-flavanoid and high-colour, Cultivar B is
low on both, Cultivar A is high-flavanoid with moderate colour. This is a two-feature visual confirmation of
the boxplot finding and the single most useful chart in this notebook.

### Q6. Is anything drifting over the sampling period?
A sanity check - a trend over `batch_date` would suggest a measurement process issue
rather than a property of the wine.

In [ ]:
monthly = (df.set_index("batch_date").sort_index()
           .resample("ME")[["alcohol", "magnesium"]].mean())
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly.index, monthly["alcohol"], marker="o", label="alcohol")
ax2 = ax.twinx()
ax2.plot(monthly.index, monthly["magnesium"], marker="s", color="#c44e52", label="magnesium")
ax.set_ylabel("mean alcohol"); ax2.set_ylabel("mean magnesium")
ax.set_title("Monthly mean by batch date"); fig.legend(loc="upper right")
plt.tight_layout(); plt.savefig(f"{VIZ}/06_trend_by_batch_date.png", dpi=120); plt.close()
print(monthly.round(2))

**Interpretation.** The monthly means fluctuate within a narrow band with no direction
to them - consistent with random batch-to-batch variation rather than instrument
drift. The `batch_date` column therefore carries no usable signal and can be left out
of modelling. Recording a negative result like this is still worth doing: it rules
out a confound instead of leaving it unexamined.

## 5. Save the cleaned dataset
Written to a **new** file - the raw export is never overwritten.

In [ ]:
out_path = "dataset/wine_quality_cleaned.csv"
df.to_csv(out_path, index=False)
print(f"saved {out_path}  shape={df.shape}")
print(df.dtypes)

## 6. Findings and limitations

**Findings**
1. The raw export contained 6 duplicate rows, 12 label spellings for 3 real
   categories, a numeric column stored as text, mixed date formats and 5 chemically
   impossible values - none of which would have raised an error, all of which would
   have quietly corrupted a model.
2. `flavanoids`, `proline_mg_l` and `color_intensity` separate the three cultivars
   almost completely; `flavanoids` alone does most of the work.
3. `total_phenols`, `flavanoids` and `od280_od315` are strongly correlated and carry
   largely the same information.
4. Several columns are right-skewed, which is why median imputation was used.
5. No time trend exists across batch dates, so that column is not a confound.

**Limitations**
- 178 samples is small; the group medians used for imputation rest on as few as
  ~48 rows for Cultivar C.
- Imputed cells are flagged by `was_imputed` but the imputation still slightly
  reduces true variance in `alcohol`, `magnesium` and `hue`.
- The injected data-quality problems are documented and reproducible, but they are
  my simulation of a real export rather than errors found in the wild.